In [2]:
import numpy as np
import tiktoken
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.nn.functional as F

In [3]:
with open('./data/shakespeare/input.txt','r',encoding='utf-8') as f:
    content = f.read()

enc = tiktoken.encoding_for_model("gpt2")
vocab_size = len(enc._mergeable_ranks)
sequence_size = 256
batch_size = 256

In [4]:
# Want to encode the data with Tokenizer from tiktoken library
# Need to split into batches
# Goal is sequence size - 1 number of tokens and goal is to predict the next token
def create_batch(tokens,sequence_length,batch_size):
    # Remove characters that don't fit beyond sequence lenght
    torch_tokens = torch.tensor(tokens).unsqueeze(-1)
    remainder = torch_tokens.shape[0] % sequence_length
    torch_tokens = torch_tokens[remainder:]
    # Split matrix into input and target characters
    sequence_matrix = torch_tokens.reshape(-1,sequence_length)
    train_portion = sequence_matrix[:,0:sequence_length-1]
    target_portion = sequence_matrix[:,sequence_length-1:]
    
    return train_portion[0:batch_size], target_portion[0:batch_size]
    
encoder_content = enc.encode(content)
train_b, target_b = create_batch(encoder_content,sequence_size,8)

# Lets create a dataset

In [5]:
# Lets create a dataset class for the above function
# Input data should already be tokenized
# sequence_length is the number of tokens used in the input
# batch size is how many sequences are used at the same time
class customShakespeare(Dataset):
    def __init__(self,data,sequence_length,mode='train'):
        self.text = torch.tensor(data).reshape(len(data),-1)
        self.seq_length = sequence_length
        self.process_data()
    
    def process_data(self):
        torch_tokens = torch.tensor(self.text).unsqueeze(-1)
        remainder = torch_tokens.shape[0] % self.seq_length
        self.text = torch_tokens[remainder:].reshape(-1,self.seq_length)
    
    def __len__(self):
        return self.text.shape[0]
    
    def __getitem__(self,idx):
        train_portion = self.text[idx,0:self.seq_length-1]
        target_portion = self.text[idx,self.seq_length-1:]
        return train_portion, target_portion

dataset = customShakespeare(encoder_content,128)
train_dataloader = DataLoader(dataset, batch_size=64, shuffle=True)
train_feat, targets = next(iter(train_dataloader))
print(train_feat.shape)


torch.Size([64, 127])


/tmp/ipykernel_9116/1350304272.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch_tokens = torch.tensor(self.text).unsqueeze(-1)


In [ ]:
# Lets start creating the GPT2 model step by step
import math
class MaskedMultiAttention(torch.nn.Module):
    def __init__(self,n_heads,dimension,dropout = 0.2):
        super(MaskedMultiAttention, self).__init__()
        self.heads = n_heads
        self.dim = dimension
        self.Q = torch.nn.Linear(self.dim,self.dim)
        self.K = torch.nn.Linear(self.dim,self.dim)
        self.V = torch.nn.Linear(self.dim,self.dim)
        self.out_proj = torch.nn.Linear(self.dim,self.dim)
        self.dropout = dropout
    
    def forward(self, x):
        q = self.Q(x).view(x.shape[0],x.shape[1],self.heads,self.dim//self.heads).transpose(1,2)
        k = self.K(x).view(x.shape[0],x.shape[1],self.heads,self.dim//self.heads).transpose(1,2)
        v = self.V(x).view(x.shape[0],x.shape[1],self.heads,self.dim//self.heads).transpose(1,2)
        seq_len = q.shape[2]
        # Attention calculation
        scores = torch.matmul(q,k.transpose(2,3)) / math.sqrt(self.dim//self.heads)

        mask = torch.tril(torch.ones((seq_len,seq_len))).cuda()
        scores = scores.masked_fill(mask == 0,float('-inf'))
        attn_weights = F.softmax(scores,dim=-1)
        

        if self.dropout:
            attn_weights = F.dropout(attn_weights,p=self.dropout)
        
        context = torch.matmul(attn_weights,v).transpose(1,2).contiguous()
        context = context.view(x.shape[0],x.shape[1],self.dim)

        return self.out_proj(context)

class Layer(torch.nn.Module):
    def __init__(self,n_heads,dimension,dropout = 0.2):
        super(Layer, self).__init__()
        self.mha = MaskedMultiAttention(n_heads,dimension,dropout)
        self.l_norm  = torch.nn.LayerNorm(dimension)
    def forward(self,x):
        mha = self.mha(x)
        s = x + mha 
        return self.l_norm(s)


class GPT2(torch.nn.Module):
    def __init__(self,d,vocab_size,dropout=0.2,num_heads =4,n_layers=3):
        super(GPT2, self).__init__()
        self.dimension = d
        self.vocab_size = vocab_size
        self.dropout = dropout
        self.num_heads = num_heads
        self.n_layers = n_layers
        self.embedding = torch.nn.Embedding(self.vocab_size,self.dimension)
        self.layers = torch.nn.Sequential(Layer(self.num_heads,self.dimension))
        for i in range(1,self.n_layers):
            self.layers.append(Layer(self.num_heads,self.dimension))
        self.prediction = torch.nn.Linear(self.dimension,vocab_size)
    def positional_encoding(self,x):
        # Taken from pytorch
        position = torch.arange(self.vocab_size).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, self.dimension, 2) * (-math.log(10000.0) / self.dimension))
        term = position * div_term
        
        pe = torch.zeros(self.vocab_size,1,self.dimension).cuda()
        pe[:,0,0::2] = torch.sin(term)
        pe[:,0,1::2] = torch.cos(term)
        return pe
    

    def forward(self, x):
        t_e = self.embedding(x) # Output = (seq_length, batch_size, dimension_size)
        p_e = t_e + self.positional_encoding(t_e)[:t_e.shape[0]] # Same output as above, but specify a position vector for each embedding
        out = self.layers(p_e)
        pred = F.softmax(self.prediction(out),dim=-1)
        
        return pred

model = GPT2(768,vocab_size).cuda()
train_feat, targets = next(iter(train_dataloader))

'''
train_feat = train_feat.cuda()
targets = targets.cuda().squeeze()
output = model(train_feat)
next_token_logits = output[:, -1, :]
'''
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.Ada


torch.Size([64])
torch.Size([64, 50256])
tensor(10.8249, device='cuda:0', grad_fn=<NllLossBackward0>)


In [ ]:
for x,y in train_dataloader: